# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/viki22uied/ML/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Check two signals, then state my rule

**Lane: Refresh / Content Opportunity Scoring. Build window: `month=2026-03` (mid-panel, not the
final month).**

I check two signals before I trust them in a rule:

1. **Staleness** — behind FlyRank's real refresh flags. FlyRank flags a page for refresh when it
   is old AND still visible (still getting traffic). A stale page nobody sees is a prune
   candidate, not a refresh candidate — the flag only fires when both are true.
2. **CTR vs. position** — behind FlyRank's real CTR-fix logic. Every position has a typical CTR.
   A page stuck below what pages at its own position normally get is a CTR-fix candidate. This
   only makes sense if CTR truly drops as position gets worse — I check that first.

Each signal gets one bucket table with `n` per bucket, then one verdict word: CONFIRMED, OPPOSITE,
MIXED, or FALSE.

In [1]:
# Setup: connect DuckDB to the hosted warehouse. Never paste the token into a cell (public repo) —
# use an env var or Colab Secret named HF_TOKEN, prompt only as a last resort.
import os, getpass
import duckdb
import pandas as pd

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
MARCH = f"{REL}/fact_content_daily_performance/month=2026-03/*.parquet"
DIM_CONTENT = f"{REL}/dim_content.parquet"

# One content-level table for the whole notebook, built only from month=2026-03.
# gsc_avg_position == 0 marks "no rank data" on that day, so it is excluded from the average,
# the same way avg_position == 0 works in the starter CSV.
base = con.sql(f"""
    WITH agg AS (
        SELECT content_hash_id,
               SUM(gsc_impressions) AS imp_march,
               SUM(gsc_clicks) AS clk_march,
               AVG(CASE WHEN gsc_avg_position > 0 THEN gsc_avg_position END) AS avg_pos_march
        FROM read_parquet('{MARCH}')
        WHERE gsc_data_available IS TRUE
        GROUP BY 1
        HAVING imp_march >= 50
    )
    SELECT a.content_hash_id, a.imp_march, a.clk_march, a.avg_pos_march,
           DATE_DIFF('day', d.content_created_date, DATE '2026-03-31') AS content_age_days
    FROM agg a
    JOIN read_parquet('{DIM_CONTENT}') d USING (content_hash_id)
    WHERE d.content_created_date IS NOT NULL
""").df()
base = base.dropna(subset=['avg_pos_march']).reset_index(drop=True)
base['ctr_march'] = base['clk_march'] / base['imp_march']

print(f'{len(base):,} content items, month=2026-03 only')
base.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

116,113 content items, month=2026-03 only


,content_hash_id,imp_march,clk_march,avg_pos_march,content_age_days,ctr_march
0,content_2edf5c5651a9646a,91.0,7.0,4.089909,242,0.076923
1,content_baf127a330a8133d,61.0,0.0,4.611861,237,0.000000
2,content_a9a07c4648f79b56,59.0,0.0,18.137037,232,0.000000
3,content_38f4a52d8909895e,102.0,2.0,5.962812,226,0.019608
4,content_2e296120acb03e93,3146.0,0.0,60.451235,375,0.000000


### Signal 1 — staleness (behind the refresh flags)

`stale` = the page is 180+ days old. `visible` = it still got 500+ impressions in March. Four
buckets: stale × visible.

In [2]:
base['stale'] = base['content_age_days'] >= 180
base['visible'] = base['imp_march'] >= 500

signal1_table = base.groupby(['stale', 'visible']).size().rename('n').reset_index()
print(signal1_table)
print()

within_visible = base[base['visible']]
print('mean CTR, visible pages, by staleness:')
print(within_visible.groupby('stale')['ctr_march'].mean())
print()
print('mean avg position, visible pages, by staleness:')
print(within_visible.groupby('stale')['avg_pos_march'].mean())

   stale  visible      n
0  False    False  26291
1  False     True  30460
2   True    False  27898
3   True     True  31464

mean CTR, visible pages, by staleness:
stale
False    0.003082
True     0.002579
Name: ctr_march, dtype: float64

mean avg position, visible pages, by staleness:
stale
False    11.632012
True     11.651719
Name: avg_pos_march, dtype: float64


**Verdict: MIXED.**

Among visible pages, stale pages do get a lower average CTR than fresh pages (about 0.26% vs.
0.31%, a real but small gap). But average position barely moves (about 11.65 vs. 11.63) — staleness
does not predict a worse rank at all. One metric leans the way I expected, the other shows nothing.
I call this MIXED, not CONFIRMED, because I only got a partial, weak match.

A weak or mixed result is still a good outcome here. It stops me from writing "stale pages always
rank worse" into a rule when the data does not back that claim. Staleness stays in the rule only as
a small secondary signal, not the main driver — see Section 2.

### Signal 2 — CTR vs. position (behind the CTR-fix logic)

First check the assumption the flag depends on: does CTR really drop as position gets worse? Bucket
positions into four tiers, then split each tier at its own median CTR.

In [3]:
def pos_bucket(p):
    if p <= 3:
        return '1_top3'
    if p <= 10:
        return '2_page1'
    if p <= 20:
        return '3_page2'
    return '4_beyond'

base['pos_bucket'] = base['avg_pos_march'].apply(pos_bucket)
base['bucket_median_ctr'] = base.groupby('pos_bucket')['ctr_march'].transform('median')
base['ctr_below_bucket_median'] = base['ctr_march'] < base['bucket_median_ctr']

print('median CTR per position bucket (does it fall as position gets worse?):')
print(base.groupby('pos_bucket')['ctr_march'].median())
print()

signal2_table = base.groupby(['pos_bucket', 'ctr_below_bucket_median']).size().rename('n').reset_index()
print(signal2_table)

median CTR per position bucket (does it fall as position gets worse?):
pos_bucket
1_top3      0.002466
2_page1     0.001727
3_page2     0.000561
4_beyond    0.000000
Name: ctr_march, dtype: float64

  pos_bucket  ctr_below_bucket_median      n
0     1_top3                    False   4286
1     1_top3                     True   4285
2    2_page1                    False  25771
3    2_page1                     True  25754
4    3_page2                    False  12309
5    3_page2                     True  12308
6   4_beyond                    False  31400


**Verdict: CONFIRMED.**

Median CTR falls every step as position gets worse: about 0.25% in the top 3, 0.17% on page one,
0.06% on page two, and 0.00% beyond that. Position really does set an expected CTR level. That
means "below your own position tier's median CTR" is a real, meaningful group to flag — not noise.
The `4_beyond` tier has a median of exactly 0, so no page there can sit "below" it; that tier
naturally produces zero CTR-fix candidates, which is correct, not a bug.

This confirms the assumption behind FlyRank's CTR-fix logic: comparing a page's CTR to its own
position peers, not to all pages, is the right way to spot underperformance.

### My rule, in plain words

A page is worth reviewing first if it already gets real traffic, and its CTR sits below what pages
at its own position normally get, and that gap is worth the most clicks if fixed. Old-and-visible
pages move up a little, since staleness showed a small real gap in Section 1.

**Score** = `imp_march × ctr_gap × stale_boost`, where `ctr_gap = max(bucket_median_ctr - ctr_march, 0)`
(the CTR points a page is missing versus its position peers) and `stale_boost` is 1.25 if the page
is stale and visible, else 1.0. This is the estimated extra clicks the page could pick up by closing
its CTR gap, lightly boosted for pages that are also overdue for a look.

**Reason code** (one, applied to every scored row): `ctr_below_position_median`.

**Action label** (one, applied to every scored row): `review_for_ctr_fix`.

## 2. Build the ranked queue (writes the CSV)

Score everything, rank it, write `work/outputs/baseline_action_score.csv` — from this notebook,
nowhere else. Before writing, check the rule uses no future-window input and no label-derived
input.

In [4]:
base['ctr_gap'] = (base['bucket_median_ctr'] - base['ctr_march']).clip(lower=0)
base['stale_boost'] = base.apply(lambda r: 1.25 if (r['stale'] and r['visible']) else 1.0, axis=1)
base['score'] = base['imp_march'] * base['ctr_gap'] * base['stale_boost']
base['reason_code'] = 'ctr_below_position_median'
base['action_label'] = 'review_for_ctr_fix'

# Rule-input check, before writing anything: every input column must come from month=2026-03
# facts or a static dim_content field, and none may be derived from a label.
inputs_used = ['imp_march', 'avg_pos_march', 'ctr_march', 'pos_bucket', 'bucket_median_ctr',
               'content_age_days', 'stale', 'visible']
future_window_terms = ('sample', 'june', '2026-04', '2026-05', '2026-06')
label_terms = ('label', 'declining', 'is_declining', 'needs_review')
assert not any(t in ' '.join(inputs_used).lower() for t in future_window_terms), 'future-window input found'
assert not any(t in ' '.join(inputs_used).lower() for t in label_terms), 'label-derived input found'
print('rule-input check passed: no future-window input, no label-derived input')

queue = base.sort_values('score', ascending=False).reset_index(drop=True)
queue.insert(0, 'rank', queue.index + 1)

out_cols = ['rank', 'content_hash_id', 'imp_march', 'avg_pos_march', 'ctr_march', 'pos_bucket',
            'bucket_median_ctr', 'ctr_gap', 'stale', 'visible', 'score', 'reason_code', 'action_label']
os.makedirs('work/outputs', exist_ok=True)
queue[out_cols].to_csv('work/outputs/baseline_action_score.csv', index=False)
print(f'wrote work/outputs/baseline_action_score.csv, {len(queue):,} rows')
queue[out_cols].head()

rule-input check passed: no future-window input, no label-derived input


wrote work/outputs/baseline_action_score.csv, 116,113 rows


,rank,content_hash_id,imp_march,avg_pos_march,ctr_march,pos_bucket,bucket_median_ctr,ctr_gap,stale,visible,score,reason_code,action_label
0,1,content_44f34c0a90047651,212404.0,7.346909,0.000113,2_page1,0.001727,0.001614,False,True,342.846287,ctr_below_position_median,review_for_ctr_fix
1,2,content_8e1334d6356668e3,134984.0,4.545582,0.000007,2_page1,0.001727,0.001720,True,True,290.166235,ctr_below_position_median,review_for_ctr_fix
2,3,content_fec55986a1868d62,124075.0,9.385150,0.000008,2_page1,0.001727,0.001719,True,True,266.614853,ctr_below_position_median,review_for_ctr_fix
3,4,content_8d7d99f109e19aa2,203497.0,2.563756,0.001420,1_top3,0.002466,0.001046,True,True,266.052713,ctr_below_position_median,review_for_ctr_fix
4,5,content_34a70fea29d15f24,143019.0,3.219473,0.000301,2_page1,0.001727,0.001426,True,True,255.012953,ctr_below_position_median,review_for_ctr_fix


## 3. Top-10 review

For each of the top 10 rows: the action, why it is in the top 10, and what would make this row
wrong.

In [5]:
top10 = queue[out_cols].head(10).copy()
top10

,rank,content_hash_id,imp_march,avg_pos_march,ctr_march,pos_bucket,bucket_median_ctr,ctr_gap,stale,visible,score,reason_code,action_label
0,1,content_44f34c0a90047651,212404.0,7.346909,0.000113,2_page1,0.001727,0.001614,False,True,342.846287,ctr_below_position_median,review_for_ctr_fix
1,2,content_8e1334d6356668e3,134984.0,4.545582,0.000007,2_page1,0.001727,0.001720,True,True,290.166235,ctr_below_position_median,review_for_ctr_fix
2,3,content_fec55986a1868d62,124075.0,9.385150,0.000008,2_page1,0.001727,0.001719,True,True,266.614853,ctr_below_position_median,review_for_ctr_fix
3,4,content_8d7d99f109e19aa2,203497.0,2.563756,0.001420,1_top3,0.002466,0.001046,True,True,266.052713,ctr_below_position_median,review_for_ctr_fix
4,5,content_34a70fea29d15f24,143019.0,3.219473,0.000301,2_page1,0.001727,0.001426,True,True,255.012953,ctr_below_position_median,review_for_ctr_fix
5,6,content_f6116743b00afc2d,107584.0,9.536301,0.000139,2_page1,0.001727,0.001588,True,True,213.512522,ctr_below_position_median,review_for_ctr_fix
6,7,content_306bc78dff1eb683,80821.0,1.488604,0.000433,1_top3,0.002466,0.002033,True,True,205.389951,ctr_below_position_median,review_for_ctr_fix
7,8,content_cd3d932d4e1c8db0,89332.0,7.786219,0.000045,2_page1,0.001727,0.001682,True,True,187.858377,ctr_below_position_median,review_for_ctr_fix
8,9,content_c46df0fa61530d86,70398.0,1.556258,0.000597,1_top3,0.002466,0.001869,True,True,164.509864,ctr_below_position_median,review_for_ctr_fix
9,10,content_fc67675904376267,60172.0,2.261303,0.000299,1_top3,0.002466,0.002167,True,True,162.987053,ctr_below_position_median,review_for_ctr_fix


In [6]:
for _, r in top10.iterrows():
    print(f"--- rank {r['rank']} ({r['content_hash_id']}) ---")
    print(f"action: {r['action_label']} (reason: {r['reason_code']})")
    print(
        f"why in the top 10: it has {r['imp_march']:,.0f} impressions in March, "
        f"sits in position bucket {r['pos_bucket']} (avg position {r['avg_pos_march']:.1f}), "
        f"and its CTR ({r['ctr_march']:.3%}) is below the {r['bucket_median_ctr']:.3%} median for "
        f"that bucket — a gap of {r['ctr_gap']:.3%}, worth about {r['score']:,.0f} extra clicks "
        f"at March volume." + (" It is also stale and visible, so its score got the 1.25x boost."
        if r['stale'] and r['visible'] else "")
    )
    print(
        "what would make this row wrong: if March was a one-off spike or dip for this page, the "
        "CTR gap would not hold in a normal month; if the page's real intent differs from most "
        "pages in its position bucket (a different content_type or SERP feature), its 'expected' "
        "CTR baseline would not apply to it."
    )
    print()

--- rank 1 (content_44f34c0a90047651) ---
action: review_for_ctr_fix (reason: ctr_below_position_median)
why in the top 10: it has 212,404 impressions in March, sits in position bucket 2_page1 (avg position 7.3), and its CTR (0.011%) is below the 0.173% median for that bucket — a gap of 0.161%, worth about 343 extra clicks at March volume.
what would make this row wrong: if March was a one-off spike or dip for this page, the CTR gap would not hold in a normal month; if the page's real intent differs from most pages in its position bucket (a different content_type or SERP feature), its 'expected' CTR baseline would not apply to it.

--- rank 2 (content_8e1334d6356668e3) ---
action: review_for_ctr_fix (reason: ctr_below_position_median)
why in the top 10: it has 134,984 impressions in March, sits in position bucket 2_page1 (avg position 4.5), and its CTR (0.001%) is below the 0.173% median for that bucket — a gap of 0.172%, worth about 290 extra clicks at March volume. It is also stale a

## 4. Weak picks + leakage check

Which top-10 picks look weak or borderline, and why? Then confirm no product flags and no
future-window data leaked into the rule.

In [7]:
# Hand review: which top-10 rows have almost zero clicks, not just a below-median CTR?
# A page with 100k+ impressions and 1-2 clicks the whole month is not a normal "underperforming"
# page — it looks more like a tracking or indexing problem than a CTR problem a title/meta
# rewrite would fix.
top10 = top10.assign(clk_march_est=lambda d: (d['imp_march'] * d['ctr_march']).round(1))
print(top10[['rank', 'content_hash_id', 'imp_march', 'ctr_march', 'clk_march_est']].to_string(index=False))

 rank          content_hash_id  imp_march  ctr_march  clk_march_est
    1 content_44f34c0a90047651   212404.0   0.000113           24.0
    2 content_8e1334d6356668e3   134984.0   0.000007            1.0
    3 content_fec55986a1868d62   124075.0   0.000008            1.0
    4 content_8d7d99f109e19aa2   203497.0   0.001420          289.0
    5 content_34a70fea29d15f24   143019.0   0.000301           43.0
    6 content_f6116743b00afc2d   107584.0   0.000139           15.0
    7 content_306bc78dff1eb683    80821.0   0.000433           35.0
    8 content_cd3d932d4e1c8db0    89332.0   0.000045            4.0
    9 content_c46df0fa61530d86    70398.0   0.000597           42.0
   10 content_fc67675904376267    60172.0   0.000299           18.0


**Weak picks: ranks 2 and 3.**

Rank 2 has about 135,000 impressions and an estimated 1 click all month. Rank 3 has about 124,000
impressions and roughly 1 click. A CTR that low, at that much volume, does not look like a normal
"page ranks fine but the snippet is weak" case — a real page at that position almost always earns
more than one click in a month. It looks more like a tracking gap, a redirect, or a page that
briefly broke, not a CTR-fix candidate. A title or meta description rewrite would not fix a
tracking problem, so these two rows need a manual look before anyone spends review time on them.

The mechanical check below (CTR gap as a share of the bucket median) does not catch this on its
own — every top-10 row clears that bar. The near-zero click count only shows up by looking at the
actual numbers by hand, which is the point of this review.

In [8]:
# Mechanical check: rows where the score is mostly volume, not a real gap (a tiny ctr_gap
# relative to the bucket's own CTR, riding on a huge imp_march to reach the top 10).
top10 = top10.assign(
    gap_share_of_median=lambda d: d['ctr_gap'] / d['bucket_median_ctr'].replace(0, pd.NA)
)
weak = top10[top10['gap_share_of_median'] < 0.15]

if len(weak) == 0:
    print('no additional weak picks by the gap-ratio check — every row clears 15% of its own '
          'bucket median on that measure alone. The real weak picks (ranks 2 and 3) only showed '
          'up in the near-zero-click hand review above, not in this mechanical check.')
else:
    for _, r in weak.iterrows():
        print(
            f"rank {r['rank']} ({r['content_hash_id']}) also looks borderline: its CTR gap "
            f"({r['ctr_gap']:.3%}) is only {r['gap_share_of_median']:.0%} of its bucket's median "
            f"CTR ({r['bucket_median_ctr']:.3%})."
        )
print()

# Leakage check, restated in code: list every column used to build the score and confirm the
# source of each one.
column_sources = {
    'imp_march': 'fact_content_daily_performance, month=2026-03 only',
    'avg_pos_march': 'fact_content_daily_performance, month=2026-03 only',
    'ctr_march': 'derived from imp_march and clk_march, both month=2026-03 only',
    'pos_bucket': 'derived from avg_pos_march, month=2026-03 only',
    'bucket_median_ctr': 'derived from ctr_march within month=2026-03 only',
    'content_age_days': 'dim_content.content_created_date (static, set once) vs. 2026-03-31',
    'stale': 'derived from content_age_days',
    'visible': 'derived from imp_march',
}
for col, source in column_sources.items():
    assert 'sample' not in source.lower() and 'june' not in source.lower(), f'{col} touches the sealed month'
    assert 'health_score' not in source and 'priority_score' not in source and 'action_type' not in source, f'{col} reuses a product flag'
    print(f'{col:20} <- {source}')
print()
print('confirmed: every input is month=2026-03 or a static, pre-March content attribute.')
print('confirmed: no FlyRank product flag (health_score, priority_score, action_type) was reused as an input.')

no additional weak picks by the gap-ratio check — every row clears 15% of its own bucket median on that measure alone. The real weak picks (ranks 2 and 3) only showed up in the near-zero-click hand review above, not in this mechanical check.

imp_march            <- fact_content_daily_performance, month=2026-03 only
avg_pos_march        <- fact_content_daily_performance, month=2026-03 only
ctr_march            <- derived from imp_march and clk_march, both month=2026-03 only
pos_bucket           <- derived from avg_pos_march, month=2026-03 only
bucket_median_ctr    <- derived from ctr_march within month=2026-03 only
content_age_days     <- dim_content.content_created_date (static, set once) vs. 2026-03-31
stale                <- derived from content_age_days
visible              <- derived from imp_march

confirmed: every input is month=2026-03 or a static, pre-March content attribute.
confirmed: no FlyRank product flag (health_score, priority_score, action_type) was reused as an input.

## Self-check

Before you submit, confirm each line honestly:

- [x] Two signal verdicts exist, with visible bucket tables and n for each.
- [x] At least one signal is linked to a real FlyRank flag (both are: staleness -> refresh flags,
      CTR vs. position -> CTR-fix logic).
- [x] One rule exists, with a score, one reason code, one action label.
- [x] The ranked queue is written from the notebook, to the exact path
      `work/outputs/baseline_action_score.csv`.
- [x] Ten rows are reviewed, with a "what would make it wrong" line for each.
- [x] No future-window input is used. No label-derived input is used.